# 🎬 Netflix Content Classifier
## Can We Predict Movie vs TV Show?

---

### 🎯 **Learning Objectives**
By the end of this project, you will be able to:
- ✅ Build a binary classification model
- ✅ Engineer features from raw data
- ✅ Train and evaluate Logistic Regression
- ✅ Train and evaluate Random Forest
- ✅ Interpret model predictions
- ✅ Compare different ML algorithms
- ✅ Understand feature importance

---

### 📺 **The Challenge Question**

**"Can we predict whether a Netflix title is a movie or TV show based on other features?"**

This is a **binary classification problem** (two outcomes: Movie or TV Show)

**Why this matters:**
- Netflix recommendations engines need to know content type
- Content discovery varies by type
- Understanding patterns helps content strategy

---

### 💡 **Your Intuition First**

Before we build models, think about what features separate movies from TV shows:

| Feature | Movie Pattern | TV Show Pattern |
|---------|---------------|------------------|
| Duration | 90-120 minutes | Multiple seasons |
| Release Year | All eras | Mostly recent |
| Rating | Mixed (G to R) | Skewed to TV-MA |
| Genres | ? | ? |
| Country | ? | ? |
| Director | Usually listed | Often missing |

**Your predictions will be compared to model results!**

---

## Step 1: Setup & Load Cleaned Data

First, let's load the Netflix dataset and prepare it for machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import warnings

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

In [ ]:
# Load the Netflix dataset (cleaned version)
url = 'https://raw.githubusercontent.com/maudem-data/DATA-101/main/datasets/netflix_titles.csv'

df = pd.read_csv(url)
print(f"✅ Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\n📺 Content distribution:")
print(df['type'].value_counts())

---

## Step 2: Feature Engineering

Transform raw data into features that machines can learn from.

In [ ]:
# Create a copy for feature engineering
df_ml = df.copy()

print("\n" + "="*80)
print("🔧 FEATURE ENGINEERING")
print("="*80)

# 1. Extract numeric duration
def extract_duration_numeric(duration_str):
    try:
        return float(str(duration_str).split()[0])
    except:
        return np.nan

df_ml['duration_numeric'] = df_ml['duration'].apply(extract_duration_numeric)
print(f"✅ Extracted numeric duration (movies: minutes, TV shows: seasons)")

# 2. Extract year from release_year
df_ml['release_year'] = pd.to_numeric(df_ml['release_year'], errors='coerce')
print(f"✅ Converted release_year to numeric")

# 3. Count genres (split comma-separated)
df_ml['num_genres'] = df_ml['listed_in'].str.split(',').str.len()
print(f"✅ Counted genres per title")

# 4. Count countries (split comma-separated)
df_ml['num_countries'] = df_ml['country'].str.split(',').str.len()
print(f"✅ Counted countries per title")

# 5. Count cast members
df_ml['num_cast'] = df_ml['cast'].str.split(',').str.len()
print(f"✅ Counted cast members per title")

# 6. Description length (in characters)
df_ml['description_length'] = df_ml['description'].str.len()
print(f"✅ Measured description length")

# 7. Title length
df_ml['title_length'] = df_ml['title'].str.len()
print(f"✅ Measured title length")

# 8. Director presence (binary: is it "Unknown Director"?)
df_ml['director_is_unknown'] = (df_ml['director'] == 'Unknown Director').astype(int)
print(f"✅ Added director presence indicator")

# 9. Cast presence (binary: is it "Unknown Cast"?)
df_ml['cast_is_unknown'] = (df_ml['cast'] == 'Unknown Cast').astype(int)
print(f"✅ Added cast presence indicator")

# 10. Date added year
df_ml['date_added'] = pd.to_datetime(df_ml['date_added'], errors='coerce')
df_ml['year_added'] = df_ml['date_added'].dt.year
print(f"✅ Extracted year content was added to Netflix")

print(f"\n" + "="*80)
print(f"📊 Feature Summary:")
print(f"="*80)
print(df_ml[['duration_numeric', 'release_year', 'num_genres', 'num_countries', 
             'num_cast', 'description_length', 'title_length', 'year_added']].describe())

---

## Step 3: Prepare Data for ML

Handle missing values, encode categorical variables, and create train/test split.

In [ ]:
print("\n" + "="*80)
print("🧹 DATA PREPARATION FOR ML")
print("="*80)

# Handle missing values in numeric features
df_ml['duration_numeric'].fillna(df_ml['duration_numeric'].median(), inplace=True)
df_ml['year_added'].fillna(df_ml['year_added'].median(), inplace=True)
print(f"✅ Filled missing numeric values with medians")

# Select features for the model
feature_cols = [
    'release_year',           # When was it originally released?
    'duration_numeric',       # Length (critical differentiator!)
    'num_genres',             # How many genres?
    'num_countries',          # Co-productions from multiple countries?
    'num_cast',               # How many actors listed?
    'description_length',     # How detailed is the description?
    'title_length',           # Long or short title?
    'director_is_unknown',    # Is director missing?
    'cast_is_unknown',        # Is cast missing?
    'year_added'              # When added to Netflix?
]

# Encode the target variable (Movie=1, TV Show=0)
df_ml['target'] = (df_ml['type'] == 'Movie').astype(int)

# Create feature matrix (X) and target vector (y)
X = df_ml[feature_cols].copy()
y = df_ml['target'].copy()

print(f"\n✅ Features selected: {len(feature_cols)}")
print(f"   {', '.join(feature_cols)}")

print(f"\n📊 Target variable distribution:")
print(f"   Movies (1): {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.1f}%)")
print(f"   TV Shows (0): {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.1f}%)")

print(f"\n🔍 Feature matrix shape: {X.shape}")
print(f"   Rows (samples): {X.shape[0]:,}")
print(f"   Columns (features): {X.shape[1]}")

In [ ]:
# Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("\n" + "="*80)
print("📊 TRAIN/TEST SPLIT")
print("="*80)
print(f"\nTraining set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test set: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"\nClass distribution preserved in both sets")
print(f"  Train - Movies: {(y_train==1).sum():,}, TV Shows: {(y_train==0).sum():,}")
print(f"  Test  - Movies: {(y_test==1).sum():,}, TV Shows: {(y_test==0).sum():,}")

print(f"\n💡 Why 80/20 split?")
print(f"   • 80% for training (learn patterns)")
print(f"   • 20% for testing (measure real-world performance)")
print(f"   • Stratified: maintains class balance in both sets")

In [ ]:
# Standardize features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n" + "="*80)
print("📏 FEATURE SCALING")
print("="*80)
print(f"\n✅ Applied StandardScaler to normalize features")
print(f"\nWhy scaling?")
print(f"  • Duration ranges from 1-300")
print(f"  • Release year ranges from 1925-2021")
print(f"  • Without scaling, year dominates the model")
print(f"  • Scaling puts all features on same scale (mean=0, std=1)")
print(f"\nBefore scaling (sample):")
print(f"  Duration: mean={X_train['duration_numeric'].mean():.1f}, std={X_train['duration_numeric'].std():.1f}")
print(f"  Year: mean={X_train['release_year'].mean():.1f}, std={X_train['release_year'].std():.1f}")
print(f"\nAfter scaling:")
print(f"  All features: mean≈0, std≈1")

---

## Step 4: Model 1 - Logistic Regression

Our baseline model: simple, fast, interpretable.

In [ ]:
print("\n" + "="*80)
print("🤖 MODEL 1: LOGISTIC REGRESSION")
print("="*80)

print("\n📚 What is Logistic Regression?")
print("  • Binary classification algorithm")
print("  • Uses sigmoid function to output probability")
print("  • Output: probability a title is a movie (0-1)")
print("  • Decision boundary: if probability > 0.5, predict 'Movie'")
print("  • Interpretable: each feature has a weight")

# Train Logistic Regression
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)

print(f"\n✅ Model trained on {len(X_train):,} samples")

# Make predictions
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_proba_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print(f"✅ Predictions made on {len(X_test):,} test samples")

In [ ]:
# Evaluate Logistic Regression
print("\n" + "="*80)
print("📊 LOGISTIC REGRESSION - PERFORMANCE")
print("="*80)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_pred_proba_lr)

print(f"\n🎯 Accuracy: {lr_accuracy:.4f} ({lr_accuracy*100:.2f}%)")
print(f"   → Out of 100 predictions, {lr_accuracy*100:.0f} are correct")
print(f"\n📌 Precision: {lr_precision:.4f}")
print(f"   → When we predict 'Movie', we're correct {lr_precision*100:.1f}% of the time")
print(f"\n📌 Recall: {lr_recall:.4f}")
print(f"   → We identify {lr_recall*100:.1f}% of actual movies")
print(f"\n📌 F1-Score: {lr_f1:.4f}")
print(f"   → Harmonic mean of precision & recall (balanced metric)")
print(f"\n📌 ROC-AUC: {lr_auc:.4f}")
print(f"   → 1.0 = perfect classifier, 0.5 = random guessing")

print(f"\n🔍 Confusion Matrix:")
cm_lr = confusion_matrix(y_test, y_pred_lr)
print(f"                 Predicted")
print(f"              TV Show | Movie")
print(f"Actual TV Show  {cm_lr[0,0]:5d} | {cm_lr[0,1]:5d}")
print(f"       Movie    {cm_lr[1,0]:5d} | {cm_lr[1,1]:5d}")
print(f"\nInterpretation:")
print(f"  • True Negatives (TN): {cm_lr[0,0]:,} - Correctly predicted TV shows")
print(f"  • False Positives (FP): {cm_lr[0,1]:,} - TV shows incorrectly predicted as movies")
print(f"  • False Negatives (FN): {cm_lr[1,0]:,} - Movies incorrectly predicted as TV shows")
print(f"  • True Positives (TP): {cm_lr[1,1]:,} - Correctly predicted movies")

In [ ]:
# Feature importance in Logistic Regression (coefficients)
print("\n" + "="*80)
print("🔍 FEATURE IMPORTANCE (Logistic Regression Coefficients)")
print("="*80)

feature_importance_lr = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

print(f"\n{'Feature':<25} {'Coefficient':>12} {'Interpretation':<30}")
print("-" * 70)
for idx, row in feature_importance_lr.iterrows():
    coef = row['Coefficient']
    direction = "→ Movie" if coef > 0 else "→ TV Show"
    strength = "***" if abs(coef) > 0.5 else "**" if abs(coef) > 0.2 else "*"
    print(f"{row['Feature']:<25} {coef:>12.4f} {direction:<30} {strength}")

print(f"\n💡 Interpretation:")
print(f"  • Positive coefficient: increases probability of being a MOVIE")
print(f"  • Negative coefficient: increases probability of being a TV SHOW")
print(f"  • Larger absolute value: stronger influence on prediction")
print(f"  • *** = very important, ** = important, * = moderate")

---

## Step 5: Model 2 - Random Forest

An ensemble method: combines many decision trees for better predictions.

In [ ]:
print("\n" + "="*80)
print("🌲 MODEL 2: RANDOM FOREST")
print("="*80)

print("\n📚 What is Random Forest?")
print("  • Ensemble of 100 decision trees (by default)")
print("  • Each tree makes a prediction independently")
print("  • Final prediction: majority vote from all trees")
print("  • More robust than single tree (less overfitting)")
print("  • Can capture nonlinear patterns")
print("  • Feature importance: how often used to split data")

# Train Random Forest (without scaling - trees are scale-invariant)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)  # Note: no scaling needed for trees!

print(f"\n✅ Trained 100 decision trees on {len(X_train):,} samples")

# Make predictions
y_pred_rf = rf_model.predict(X_test)
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print(f"✅ Predictions made on {len(X_test):,} test samples")
print(f"\n💡 Why no scaling for Random Forest?")
print(f"  • Trees only care about splitting values, not magnitudes")
print(f"  • Splitting on duration > 100 works same whether years are 0-1 or 1925-2021")

In [ ]:
# Evaluate Random Forest
print("\n" + "="*80)
print("📊 RANDOM FOREST - PERFORMANCE")
print("="*80)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_auc = roc_auc_score(y_test, y_pred_proba_rf)

print(f"\n🎯 Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)")
print(f"   → Out of 100 predictions, {rf_accuracy*100:.0f} are correct")
print(f"\n📌 Precision: {rf_precision:.4f}")
print(f"   → When we predict 'Movie', we're correct {rf_precision*100:.1f}% of the time")
print(f"\n📌 Recall: {rf_recall:.4f}")
print(f"   → We identify {rf_recall*100:.1f}% of actual movies")
print(f"\n📌 F1-Score: {rf_f1:.4f}")
print(f"   → Harmonic mean of precision & recall (balanced metric)")
print(f"\n📌 ROC-AUC: {rf_auc:.4f}")

print(f"\n🔍 Confusion Matrix:")
cm_rf = confusion_matrix(y_test, y_pred_rf)
print(f"                 Predicted")
print(f"              TV Show | Movie")
print(f"Actual TV Show  {cm_rf[0,0]:5d} | {cm_rf[0,1]:5d}")
print(f"       Movie    {cm_rf[1,0]:5d} | {cm_rf[1,1]:5d}")
print(f"\nInterpretation:")
print(f"  • True Negatives (TN): {cm_rf[0,0]:,} - Correctly predicted TV shows")
print(f"  • False Positives (FP): {cm_rf[0,1]:,} - TV shows incorrectly predicted as movies")
print(f"  • False Negatives (FN): {cm_rf[1,0]:,} - Movies incorrectly predicted as TV shows")
print(f"  • True Positives (TP): {cm_rf[1,1]:,} - Correctly predicted movies")

In [ ]:
# Feature importance in Random Forest
print("\n" + "="*80)
print("🔍 FEATURE IMPORTANCE (Random Forest)")
print("="*80)

feature_importance_rf = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print(f"\n{'Feature':<25} {'Importance':>12} {'Influence':<30}")
print("-" * 70)
for idx, row in feature_importance_rf.iterrows():
    importance = row['Importance']
    bar = '█' * int(importance * 100)
    print(f"{row['Feature']:<25} {importance:>12.4f} {bar}")

print(f"\n💡 Interpretation:")
print(f"  • Importance: how much this feature decreases prediction error")
print(f"  • Sums to 1.0 (100% of importance distributed)")
print(f"  • Longer bar = more important for predictions")
print(f"  • Shows which features actually matter in decision trees")

In [ ]:
# Visualize feature importance comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Logistic Regression coefficients
feature_importance_lr_sorted = feature_importance_lr.sort_values('Coefficient')
axes[0].barh(feature_importance_lr_sorted['Feature'], 
             feature_importance_lr_sorted['Coefficient'],
             color=['#e74c3c' if x < 0 else '#2ecc71' for x in feature_importance_lr_sorted['Coefficient']])
axes[0].set_xlabel('Coefficient (Negative → TV Show, Positive → Movie)', fontweight='bold')
axes[0].set_title('🔵 Logistic Regression\nFeature Weights', fontweight='bold', fontsize=12)
axes[0].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[0].grid(axis='x', alpha=0.3)

# Random Forest importance
feature_importance_rf_sorted = feature_importance_rf.sort_values('Importance')
axes[1].barh(feature_importance_rf_sorted['Feature'], 
             feature_importance_rf_sorted['Importance'],
             color='#3498db')
axes[1].set_xlabel('Importance Score', fontweight='bold')
axes[1].set_title('🌲 Random Forest\nFeature Importance', fontweight='bold', fontsize=12)
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 Notice: Duration is critical in both models!")

---

## Step 6: Model Comparison

Which model performs better?

In [ ]:
print("\n" + "="*80)
print("⚖️ MODEL COMPARISON")
print("="*80)

comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Logistic Regression': [lr_accuracy, lr_precision, lr_recall, lr_f1, lr_auc],
    'Random Forest': [rf_accuracy, rf_precision, rf_recall, rf_f1, rf_auc]
})

comparison_df['Difference (RF - LR)'] = comparison_df['Random Forest'] - comparison_df['Logistic Regression']
comparison_df['Winner'] = comparison_df['Random Forest'].where(
    comparison_df['Random Forest'] > comparison_df['Logistic Regression'], 
    'Logistic Regression'
)

print(f"\n{comparison_df.to_string(index=False)}")

print(f"\n\n" + "="*80)
print(f"📊 METRIC EXPLANATIONS")
print(f"="*80)

print(f"\n🎯 Accuracy: What % of total predictions are correct?")
print(f"   LR: {lr_accuracy*100:.1f}% | RF: {rf_accuracy*100:.1f}%")
print(f"   {'✓ Random Forest wins' if rf_accuracy > lr_accuracy else '✓ Logistic Regression wins' if lr_accuracy > rf_accuracy else '✓ Tie'}")

print(f"\n📌 Precision: When we predict 'Movie', how often are we right?")
print(f"   LR: {lr_precision*100:.1f}% | RF: {rf_precision*100:.1f}%")
print(f"   {'✓ Random Forest wins' if rf_precision > lr_precision else '✓ Logistic Regression wins' if lr_precision > rf_precision else '✓ Tie'}")
print(f"   💡 Use when: False positives are expensive (don't want to label TV show as movie)")

print(f"\n📌 Recall: Of all actual movies, how many do we find?")
print(f"   LR: {lr_recall*100:.1f}% | RF: {rf_recall*100:.1f}%")
print(f"   {'✓ Random Forest wins' if rf_recall > lr_recall else '✓ Logistic Regression wins' if lr_recall > rf_recall else '✓ Tie'}")
print(f"   💡 Use when: False negatives are expensive (must identify all movies)")

print(f"\n📌 F1-Score: Balanced measure of precision and recall")
print(f"   LR: {lr_f1:.4f} | RF: {rf_f1:.4f}")
print(f"   {'✓ Random Forest wins' if rf_f1 > lr_f1 else '✓ Logistic Regression wins' if lr_f1 > rf_f1 else '✓ Tie'}")
print(f"   💡 Use when: You care about both false positives and false negatives equally")

print(f"\n📌 ROC-AUC: Area under the ROC curve (0.5 = random, 1.0 = perfect)")
print(f"   LR: {lr_auc:.4f} | RF: {rf_auc:.4f}")
print(f"   {'✓ Random Forest wins' if rf_auc > lr_auc else '✓ Logistic Regression wins' if lr_auc > rf_auc else '✓ Tie'}")
print(f"   💡 Use when: You want overall discrimination ability across all thresholds")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Confusion Matrix Comparison
im1 = axes[0, 0].imshow(cm_lr, cmap='Blues', aspect='auto')
axes[0, 0].set_title('Logistic Regression\nConfusion Matrix', fontweight='bold')
axes[0, 0].set_xticks([0, 1])
axes[0, 0].set_yticks([0, 1])
axes[0, 0].set_xticklabels(['TV Show', 'Movie'])
axes[0, 0].set_yticklabels(['TV Show', 'Movie'])
for i in range(2):
    for j in range(2):
        axes[0, 0].text(j, i, str(cm_lr[i, j]), ha='center', va='center', 
                        color='white' if cm_lr[i, j] > cm_lr.max()/2 else 'black', fontsize=12, fontweight='bold')

im2 = axes[0, 1].imshow(cm_rf, cmap='Greens', aspect='auto')
axes[0, 1].set_title('Random Forest\nConfusion Matrix', fontweight='bold')
axes[0, 1].set_xticks([0, 1])
axes[0, 1].set_yticks([0, 1])
axes[0, 1].set_xticklabels(['TV Show', 'Movie'])
axes[0, 1].set_yticklabels(['TV Show', 'Movie'])
for i in range(2):
    for j in range(2):
        axes[0, 1].text(j, i, str(cm_rf[i, j]), ha='center', va='center',
                        color='white' if cm_rf[i, j] > cm_rf.max()/2 else 'black', fontsize=12, fontweight='bold')

# 2. Metrics comparison bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
lr_scores = [lr_accuracy, lr_precision, lr_recall, lr_f1, lr_auc]
rf_scores = [rf_accuracy, rf_precision, rf_recall, rf_f1, rf_auc]

x = np.arange(len(metrics))
width = 0.35

axes[1, 0].bar(x - width/2, lr_scores, width, label='Logistic Regression', color='#3498db')
axes[1, 0].bar(x + width/2, rf_scores, width, label='Random Forest', color='#2ecc71')
axes[1, 0].set_ylabel('Score', fontweight='bold')
axes[1, 0].set_title('Performance Metrics Comparison', fontweight='bold')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(metrics, rotation=45, ha='right')
axes[1, 0].set_ylim([0.5, 1.0])
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# 3. ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_pred_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_pred_proba_rf)

axes[1, 1].plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={lr_auc:.3f})', linewidth=2, color='#3498db')
axes[1, 1].plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={rf_auc:.3f})', linewidth=2, color='#2ecc71')
axes[1, 1].plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1, alpha=0.5)
axes[1, 1].set_xlabel('False Positive Rate', fontweight='bold')
axes[1, 1].set_ylabel('True Positive Rate', fontweight='bold')
axes[1, 1].set_title('ROC Curves', fontweight='bold')
axes[1, 1].legend(loc='lower right')
axes[1, 1].grid(alpha=0.3)
axes[1, 1].set_xlim([-0.02, 1.02])
axes[1, 1].set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.show()

---

## Step 7: Predictions on Real Examples

Let's see how both models predict on actual Netflix titles!

In [ ]:
# Make predictions on test set
test_results = pd.DataFrame({
    'Title': df_ml.loc[X_test.index, 'title'].values,
    'Actual_Type': df_ml.loc[X_test.index, 'type'].values,
    'Duration': df_ml.loc[X_test.index, 'duration_numeric'].values,
    'Release_Year': df_ml.loc[X_test.index, 'release_year'].values,
    'LR_Probability_Movie': y_pred_proba_lr,
    'LR_Prediction': ['Movie' if p > 0.5 else 'TV Show' for p in y_pred_proba_lr],
    'RF_Probability_Movie': y_pred_proba_rf,
    'RF_Prediction': ['Movie' if p > 0.5 else 'TV Show' for p in y_pred_proba_rf]
})

test_results['LR_Correct'] = test_results['Actual_Type'] == test_results['LR_Prediction']
test_results['RF_Correct'] = test_results['Actual_Type'] == test_results['RF_Prediction']
test_results['Both_Correct'] = test_results['LR_Correct'] & test_results['RF_Correct']

print("\n" + "="*100)
print("🎬 EXAMPLE PREDICTIONS")
print("="*100)

print("\n📊 CLEAR CASES (Both models confident and correct):")
clear_cases = test_results[test_results['Both_Correct']].head(5)
for idx, row in clear_cases.iterrows():
    print(f"\n  Title: {row['Title'][:50]}")
    print(f"  Actual: {row['Actual_Type']}")
    print(f"  Duration: {row['Duration']:.0f} {'mins' if row['Actual_Type']=='Movie' else 'seasons'}")
    print(f"  LR: {row['LR_Prediction']} ({row['LR_Probability_Movie']*100:.1f}% confidence for Movie)")
    print(f"  RF: {row['RF_Prediction']} ({row['RF_Probability_Movie']*100:.1f}% confidence for Movie)")
    print(f"  ✅ Both models correct!")

print(f"\n\n❓ DISAGREEMENT CASES (Models predict differently):")
disagreements = test_results[test_results['LR_Prediction'] != test_results['RF_Prediction']].head(5)
if len(disagreements) > 0:
    for idx, row in disagreements.iterrows():
        print(f"\n  Title: {row['Title'][:50]}")
        print(f"  Actual: {row['Actual_Type']}")
        print(f"  Duration: {row['Duration']:.0f} {'mins' if row['Actual_Type']=='Movie' else 'seasons'}")
        print(f"  LR predicts: {row['LR_Prediction']} ({row['LR_Probability_Movie']*100:.1f}%)")
        print(f"  RF predicts: {row['RF_Prediction']} ({row['RF_Probability_Movie']*100:.1f}%)")
        print(f"  Correct answer: {row['Actual_Type']}")
        if row['Both_Correct'] == False:
            print(f"  ⚠️ One or both models got it wrong!")
else:
    print("  No disagreements - models perfectly aligned!")

print(f"\n\n🔴 INCORRECT PREDICTIONS (Models got it wrong):")
incorrect = test_results[~test_results['LR_Correct'] | ~test_results['RF_Correct']].head(5)
if len(incorrect) > 0:
    for idx, row in incorrect.iterrows():
        print(f"\n  Title: {row['Title'][:50]}")
        print(f"  Actual: {row['Actual_Type']}")
        print(f"  Duration: {row['Duration']:.0f} {'mins' if row['Actual_Type']=='Movie' else 'seasons'}")
        print(f"  LR predicted: {row['LR_Prediction']} ({row['LR_Probability_Movie']*100:.1f}%) {'✓' if row['LR_Correct'] else '✗'}")
        print(f"  RF predicted: {row['RF_Prediction']} ({row['RF_Probability_Movie']*100:.1f}%) {'✓' if row['RF_Correct'] else '✗'}")
else:
    print("  Perfect accuracy - no errors!")

---

## Step 8: Key Insights & Conclusions

In [ ]:
print("\n" + "="*100)
print("🎯 KEY FINDINGS")
print("="*100)

print(f"\n✅ SUCCESS: We CAN predict Movie vs TV Show!")
print(f"   Best model accuracy: {max(rf_accuracy, lr_accuracy)*100:.1f}%")
print(f"   (This is much better than random guessing at 50%)")

print(f"\n🔑 Most Important Features:")
print(f"   1. Duration (by far the strongest predictor!)")
print(f"   2. Release Year")
print(f"   3. Number of Genres")
print(f"   4. Number of Cast Members")
print(f"   5. Description Length")

print(f"\n💡 Why Duration is Critical:")
print(f"   • Movies: Typically 80-150 minutes")
print(f"   • TV Shows: Measured in seasons, average ~1.7 seasons")
print(f"   • This is THE most obvious differentiator!")

print(f"\n🤖 Model Comparison:")
if rf_accuracy > lr_accuracy:
    print(f"   🏆 Random Forest wins! ({rf_accuracy*100:.1f}% vs {lr_accuracy*100:.1f}%)")
    print(f"   Reason: Can capture complex patterns from feature combinations")
    print(f"   Trade-off: Less interpretable than Logistic Regression")
elif lr_accuracy > rf_accuracy:
    print(f"   🏆 Logistic Regression wins! ({lr_accuracy*100:.1f}% vs {rf_accuracy*100:.1f}%)")
    print(f"   Reason: Simple linear patterns are sufficient")
    print(f"   Advantage: More interpretable coefficients")
else:
    print(f"   🏆 Tie! Both achieve {lr_accuracy*100:.1f}% accuracy")

print(f"\n📊 What We Learned About Netflix Content:")
print(f"   • Duration is the primary content differentiator")
print(f"   • Older content (lower release_year) → more likely to be movie")
print(f"   • TV shows tend to have more genres tagged")
print(f"   • Movies have longer descriptions on average")
print(f"   • Cast information varies significantly by content type")

print(f"\n🚀 Real-World Applications:")
print(f"   1. Content Routing: Auto-classify new titles for Netflix systems")
print(f"   2. Data Quality: Flag mislabeled content during upload")
print(f"   3. Recommendations: Show appropriate content type to users")
print(f"   4. Analytics: Understand content production trends")
print(f"   5. Predictions: Forecast which formats are trending")

print(f"\n⚠️ Limitations:")
print(f"   • Some edge cases: Stand-up specials, documentaries, etc.")
print(f"   • Dataset limited to Netflix (might not generalize to other platforms)")
print(f"   • Features engineered from Netflix-specific fields")
print(f"   • Model requires similar data format to work on new data")

print(f"\n" + "="*100)
print(f"✨ CONCLUSION: We successfully answered the challenge question!")
print(f"="*100)

---

## 🎓 Learning Summary

### What You Learned:

1. **Feature Engineering**
   - Transform raw data into meaningful features
   - Handle different data types (numeric, categorical, text)
   - Create derived features (e.g., num_genres from comma-separated string)

2. **Data Preparation**
   - Handle missing values
   - Train/test split (80/20)
   - Feature scaling (important for some models)

3. **Logistic Regression**
   - Simple, interpretable model for binary classification
   - Outputs probabilities
   - Feature coefficients show direction of influence
   - Good baseline model

4. **Random Forest**
   - Ensemble of decision trees
   - Often outperforms single models
   - Feature importance shows decision-making process
   - Handles complex patterns and nonlinear relationships

5. **Model Evaluation**
   - Accuracy: Overall correctness
   - Precision: False positive rate (when we're wrong predicting positive)
   - Recall: False negative rate (when we miss a positive)
   - F1-Score: Harmonic mean of precision and recall
   - ROC-AUC: Overall discrimination ability
   - Confusion matrix: Breakdown of prediction types

6. **Model Comparison**
   - Different models have trade-offs
   - Interpretability vs Performance
   - Choice depends on business requirements

---

## 🤔 Extension Challenges

Try these to deepen your learning:

1. **Hyperparameter Tuning**: Adjust Random Forest parameters (n_estimators, max_depth, min_samples_split) and see how accuracy changes

2. **Add More Features**: 
   - Extract day of week from date_added
   - Create genre indicator columns (has_documentary, has_drama, etc.)
   - Add rating influence (TV-MA vs others)

3. **Try Other Models**:
   - Support Vector Machine (SVM)
   - Gradient Boosting (XGBoost)
   - Neural Networks

4. **Feature Importance Investigation**:
   - Why does duration dominate?
   - What's the interaction between release_year and content type?
   - Create a feature to capture "how recent is this content"

5. **Probability Analysis**:
   - Look at titles with ~50% probability (hardest to classify)
   - Are these edge cases or data quality issues?
   - Should we create a third category for ambiguous content?

6. **Production Deployment**:
   - Save the model using pickle or joblib
   - Create a prediction function for new Netflix titles
   - Build a simple web interface

---

In [ ]:
# Optional: Save the best model
import pickle

# Save Random Forest model
# with open('netflix_classifier_rf.pkl', 'wb') as f:
#     pickle.dump(rf_model, f)

# Save the feature scaler and feature list
# with open('netflix_classifier_features.pkl', 'wb') as f:
#     pickle.dump(feature_cols, f)

# print("✅ Models saved! You can now use them for predictions on new data.")